Python execute their code in synchronous way by following top to down approach ( by default its blocking). call next line of code afterr executing current one

- when we  need to work in asynchronous ( non-blocking) you need to use asyncio


# synchronous ( blocking)

In [10]:
# 1- example
import time

def timelog (number,delay) :
    
    for i in range(number):
     print(f"Count: {i + 1} of {number}")
     time.sleep(delay)

    
timelog(5,15)
# 15 x 5 (second)  => execute 
print("hello")

Count: 1 of 5
Count: 2 of 5
Count: 3 of 5
Count: 4 of 5
Count: 5 of 5
hello


In [ ]:
#2 Example

with open("data_4_python.txt","r" , encoding="utf-8") as file : 
    content  = file.read()
    print(content)

India is a vast and multifaceted civilization defined by its immense geographical breadth, millenniums-spanning history, profound cultural plurality, and rapid modern transformation. Stretching from the snow-capped Himalayan ridges in the north to the tropical waters of Kanyakumari at its southern tip, the subcontinent encompasses virtually every major biome on Earth, including the arid expanse of the Thar Desert, the fertile alluvial plains of the Indo-Gangetic basin, the dense rainforests of the Western Ghats, and extensive coastlines along the Arabian Sea and the Bay of Bengal.



# asynchronous ( non-blocking)

In [11]:
#Example 1
import asyncio

async def timelog(name, number, delay):
    for i in range(number):
        print(f"[{name}] {i + 1} of {number}", flush=True)
        await asyncio.sleep(delay)

print("you called me before async function call")

task1 = asyncio.create_task(timelog("Task A", 3, 1))
task2 = asyncio.create_task(timelog("Task B", 3, 0.5))

print("you called me after async function call (running concurrently now!)")


await asyncio.gather(task1, task2)
print("All background tasks completed!")

you called me before async function call
you called me after async function call (running concurrently now!)
[Task A] 1 of 3
[Task B] 1 of 3
[Task B] 2 of 3
[Task A] 2 of 3
[Task B] 3 of 3
[Task A] 3 of 3
All background tasks completed!


In [23]:
# 2 Example
import asyncio

def _read_file(file_location):
    with open(file_location, "r", encoding="utf-8") as file:
        return file.read()

async def async_file_read(file_location):
    content = await asyncio.to_thread(_read_file, file_location)
    return content

read = asyncio.create_task(async_file_read("data_4_python.txt"))
print(f"Task status: {read}")
print("called before read.")
await asyncio.gather(read)

Task status: <Task pending name='Task-932' coro=<async_file_read() running at C:\Users\anand\AppData\Local\Temp\ipykernel_5264\2017070216.py:8>>
called before read.


["India is a vast and multifaceted civilization defined by its immense geographical breadth, millenniums-spanning history, profound cultural plurality, and rapid modern transformation. Stretching from the snow-capped Himalayan ridges in the north to the tropical waters of Kanyakumari at its southern tip, the subcontinent encompasses virtually every major biome on Earth, including the arid expanse of the Thar Desert, the fertile alluvial plains of the Indo-Gangetic basin, the dense rainforests of the Western Ghats, and extensive coastlines along the Arabian Sea and the Bay of Bengal.\n\nCivilizational Roots and History\n\nHuman civilization in India traces back to the ancient Indus Valley Civilization (mature period c. 2600–1900 BCE), which pioneered urban planning, standardized brick weights, and advanced drainage systems in sites such as Harappa, Mohenjo-daro, and Dholavira. The subsequent Vedic period laid foundational philosophical and literary traditions through the composition of 

# Generator function

generator is a function that produces a sequence of values on demand (one at a time) instead of computing everything upfront and storing it in memory. It pauses its state at yield and resumes only when asked for the next item.

The Real-World Problem: Reading Massive Files
Imagine you have a 10 GB server log file, and your machine only has 8 GB of RAM.

The Problem (Using a Regular List): If you try to read all lines into a standard list at once (file.readlines()), your program attempts to allocate 10 GB into RAM, crashing immediately with MemoryError.

The Generator Solution: A generator streams one line at a time into memory, processes it, discards it, and moves to the next. Memory usage stays constant at just a few kilobytes.

Standard List Approach (Crash):
[10 GB File] ──Load All──> [8 GB RAM] ──> 💥 MemoryError (Out of Memory)

Generator Approach (Success):
[10 GB File] ──1 Line at a time──> [RAM (KB)] ──Process & Discard──> Repeat

In [4]:
import os

sample_logs = """[2026-08-16 01:00:01] INFO  path=/home status=200 device=Desktop
[2026-08-16 01:00:02] ERROR path=/pricing status=404 device=Mobile
[2026-08-16 01:00:03] INFO  path=/about status=200 device=Mobile
[2026-08-16 01:00:04] ERROR path=/contact status=500 device=Desktop
[2026-08-16 01:00:05] ERROR path=/old-page status=404 device=Mobile
[2026-08-16 01:00:06] ERROR path=/missing status=404 device=Desktop
[2026-08-16 01:00:07] ERROR path=/api/v1/user status=404 device=Mobile
"""

with open("server.log", "w") as f:
    f.write(sample_logs.strip())

def stream_large_log(file_path):
    """Generator: Reads one line at a time from disk into RAM."""
    with open(file_path, "r") as file:
        for line in file:
            yield line.strip()


def filter_status_code(log_stream, status_code="404"):
    """Generator: Filters lines on the fly without building an intermediate list."""
    for line in log_stream:
        if f"status={status_code}" in line:
            yield line

raw_logs = stream_large_log("server.log")
failed_requests = filter_status_code(raw_logs, status_code="404")

mobile_404_count = 0
print("Matching 404 Log Line")
for error_line in failed_requests:
    print(f"Found: {error_line}")
    if "device=Mobile" in error_line:
        mobile_404_count += 1

print("\nSummary")
print(f"Total 404 errors on Mobile: {mobile_404_count}")


os.remove("server.log")

Matching 404 Log Line
Found: [2026-08-16 01:00:02] ERROR path=/pricing status=404 device=Mobile
Found: [2026-08-16 01:00:05] ERROR path=/old-page status=404 device=Mobile
Found: [2026-08-16 01:00:06] ERROR path=/missing status=404 device=Desktop
Found: [2026-08-16 01:00:07] ERROR path=/api/v1/user status=404 device=Mobile

Summary
Total 404 errors on Mobile: 3
